# 01_validate_histories

**Owner:** Data and replay. **Blueprint step:** 2. **Status:** Runs locally. Not yet run in Fabric.

Validates keys and ledger history in the raw HomeBridge (fictional) tables. It builds clean account histories, report dimensions and feed watermarks. Bad records go to quarantine and stay visible (T02). Reversals keep their audit trail (T04). Stale feeds are flagged (T03).

Input: `lakehouse/raw/` from `pl_load_raw`. Output: `lakehouse/clean/`, `lakehouse/quarantine/`. All data is synthetic.

In [ ]:
OUTPUT_ROOT = None  # default: data/output
RUN_AT = None  # default: the generator as_of time

In [ ]:
import sys
from pathlib import Path

# Local run: this notebook sits in notebooks/ and the package in data/.
# Fabric run (not yet done): attach the Lakehouse and set DATA_DIR and OUTPUT_ROOT to its Files paths.
DATA_DIR = next(p for p in [Path.cwd() / "data", Path.cwd().parent / "data"] if (p / "hbdata").exists())
sys.path.insert(0, str(DATA_DIR))
OUTPUT_ROOT = Path(OUTPUT_ROOT) if OUTPUT_ROOT else DATA_DIR / "output"
LAKEHOUSE = OUTPUT_ROOT / "lakehouse"
import json
import pandas as pd
manifest = json.loads((OUTPUT_ROOT / "exports" / "manifest.json").read_text(encoding="utf-8"))
AS_OF = pd.Timestamp(manifest["config"]["as_of"])
print("Output root:", OUTPUT_ROOT, "| generator as_of:", AS_OF)

In [ ]:
from hbdata import pipelines_local, validation

run_at = pd.Timestamp(RUN_AT) if RUN_AT else AS_OF
raw = pipelines_local.read_raw(LAKEHOUSE)
result = validation.validate(raw, run_at)
validation.write_outputs(result, LAKEHOUSE)
pd.DataFrame(result['summary']).T

## Quarantine

Each row keeps the original record as JSON. Nothing here reaches clean history or model inputs.

In [ ]:
pd.Series(result['quarantine_by_reason'], name='rows').to_frame()

In [ ]:
result['quarantine'][['dataset', 'record_key', 'mortgage_id', 'reason', 'detail']].head(15)

## Feed watermarks (T03)

In [ ]:
result['watermarks']

## Checks

In [ ]:
clean = result['clean']
assert clean['payment']['entry_id'].is_unique
assert clean['transfer_request_event']['event_id'].is_unique
assert clean['servicing_contact']['contact_id'].is_unique
q = result['quarantine']
assert not set(q[q.dataset == 'payment'].record_key) & set(clean['payment'].entry_id)
rev = clean['payment'][clean['payment'].entry_type == 'reversal']
assert rev['reversal_of'].isin(clean['payment'].entry_id).all()
print('Checks passed. Clean reversals kept with their original entries:', len(rev))